In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import pathlib
import scipy.optimize

style_file = pathlib.Path.cwd().parent / 'plot_style.mplstyle'
plt.style.use(['default', style_file])

inputpath_data = pathlib.Path.cwd() / 'input_data/ephys' 
inputpath_data.mkdir(parents=True, exist_ok=True)
savepath_data = pathlib.Path.cwd() / 'output_data/ephys' 
savepath_data.mkdir(parents=True, exist_ok=True)
savepath_fig = pathlib.Path.cwd() / 'output_fig/ephys' 
savepath_fig.mkdir(parents=True, exist_ok=True)

%matplotlib inline
%config InlineBackend.figure_format='retina'

colormap = {'rdl': '#52c3c3', 'ort':'#dd7c2c', 'rdl_ort': '#662964', 'control': '#808086'}
colormap_rdl_reducedCl = {'GABA': '#52c3c3', 'GABA_reducedCl': '#808086'}
colormap_ort_reducedCl = {'Histamine': '#dd7c2c', 'Histamine_reducedCl': '#808086'}

cm_inch = 1/2.54

In [ ]:
# Calculare Nernst potential
def Nernst(conc_out, conc_in, Temp, z):

    R = 8.31451       # Gas constant in J mol^-1 K^-1
    F = 96485.33212   # Faraday constant in C mol^-1
    T = Temp + 273.15 # Temperature in K
                      # z = Valence of the ion

    E = R*T/(z*F)*np.log(conc_out/conc_in)*1000 # in mV
    
    return E

reversal_Cl_normal = Nernst(117, 1, 20, -1) # normal Ringer and IC solution, chloride reversal potential
reversal_Cl_reduced = Nernst(14, 1, 20, -1) # reduced chloride Ringer (Na-aspartate instead of NaCl), normal IC solution
# reversal_K_normal = Nernst(3, 141, 20, 1) # normal Ringer and IC solution, potassium reversal potential
# reversal_Na_normal = Nernst(130, 1.5, 20, 1) # normal Ringer and IC solution, sodium reversal potential

In [ ]:
averaged_cells = pd.read_parquet(inputpath_data / 'ephys_preprocessed_averaged_cells.parquet')

In [ ]:
cell_counts = averaged_cells[['cell', 'transmitter', 'transmitter_mM', 'protocol', 'receptor']].drop_duplicates().reset_index()
cell_counts = pd.DataFrame(cell_counts.groupby(['protocol', 'transmitter', 'transmitter_mM', 'receptor'], dropna=False).cell.count()).rename(columns={'cell': 'cell_count'})
cell_counts

In [ ]:
cell_averages_std = pd.DataFrame(averaged_cells.groupby(['cell', 'time']).i.std()).reset_index()
cell_averages_std = cell_averages_std.rename(columns={'i': 'i_std'})

In [ ]:
# Get timestamp of the peak transmitter response for each cell, based on the highest standard deviation over all holding potentials
peak_times = []

fig, ax = plt.subplots(14,10, figsize=(20,25))

ax_x = 0
ax_y = 0

for index, recording in tqdm(cell_averages_std.groupby('cell')):
    sns.lineplot(x='time', y='i_std', data=recording, ax=ax[ax_y, ax_x])
    peak_index = recording.query("time > 1.2485 and time < 1.6").i_std.idxmax()
    peak_time = recording.loc[peak_index, 'time']
    peak_times.append([recording.cell.unique()[0], peak_time])
    ax[ax_y, ax_x].axvline(x=peak_time, c='orange')
    ax[ax_y, ax_x].set_xlim(1.2, 2)
    ax_x = ax_x + 1
    if ax_x > 9:
        ax_x = 0
        ax_y = ax_y + 1

peak_times = pd.DataFrame(peak_times, columns=['cell', 'peak_time'])

In [ ]:
# Extract data at response peaks after transmitter application
peak_data = []
for cell in tqdm(peak_times.iterrows()):
    cell_peak_time = cell[1].peak_time
    cell = cell[1].cell
    peak_data_temp = averaged_cells.query("cell == @cell and time == @cell_peak_time")
    peak_data.append(peak_data_temp)
peak_data = pd.concat(peak_data).reset_index(drop=True)

peak_i_std = []
cells_list = []

for cell in tqdm(peak_times.iterrows()):
    cell_peak_time = cell[1].peak_time
    cell = cell[1].cell
    peak_i_std_temp = cell_averages_std.query("cell == @cell and time == @cell_peak_time").i_std.values[0]
    peak_i_std.append(peak_i_std_temp)
    cells_list.append(cell)
peak_i_std = pd.DataFrame({'cell': cells_list, 'peak_i_std': peak_i_std})

In [ ]:
cells_sorted = peak_i_std.sort_values('peak_i_std', ascending=False)['cell'].reset_index(drop=True).values
sort_cells = pd.DataFrame(cells_sorted).reset_index().rename(columns={0: 'cell', 'index':'sort_order'}).to_dict(orient='dict')['cell']
sort_cells = {v: k for k, v in sort_cells.items()}

averaged_cells_sorted = averaged_cells.copy()
averaged_cells_sorted['sorting'] = averaged_cells_sorted.cell.map(sort_cells)
averaged_cells_sorted = averaged_cells_sorted.sort_values(by=['sorting', 'sweep', 'time'], ascending=[True, True, True]).reset_index(drop=True)

In [ ]:
# Averaging all cells in each condition -> used for plotting later
averaged_temp = {}
for index, column_name in tqdm(enumerate(['i', 'i_start_baseline', 'i_raw', 'Z_i', 'leak', 'pdes', 'i_abs', 'puff_time'])):
    averaged_temp[column_name] = averaged_cells.groupby(['time', 'v', 'transmitter', 'transmitter_mM', 'receptor', 'sweep', 'protocol'])[column_name].mean()
averages = pd.DataFrame(averaged_temp).reset_index()

In [ ]:
# Reversal potential calculations for cells that respond to the transmitter applications
responding_cells = peak_i_std.query("peak_i_std > 100").cell.values
query = "cell in @responding_cells"
responding_cells_data = peak_data.query(query).reset_index(drop=True)

responding_cells_data['response_sign'] = responding_cells_data.i.apply(lambda x: -1 if x < 0 else 1)
responding_cells_data['response_sign_change'] = responding_cells_data['response_sign'].diff()
zero_crossings_indices = (responding_cells_data.query("response_sign_change == 2").index - 1).union(responding_cells_data.query("response_sign_change == 2").index)
zero_crossings = responding_cells_data.iloc[zero_crossings_indices]

# Interpolate two measurements (before and after the zero crossing) to estimate the zero-crossing voltage (equilibrium potential) for each cell
cells_list = []
eq_list = []
transmitter_list = []
transmitter_mM_list = []
receptor_list = []

for cell in tqdm(zero_crossings.cell.unique()):
    to_interpolate_v = zero_crossings.query("cell == @cell")['v'].values[:]
    to_interpolate_i = zero_crossings.query("cell == @cell")['i'].values[:]
    eq = np.interp(0, to_interpolate_i, to_interpolate_v)
    transmitter = zero_crossings.query("cell == @cell")['transmitter'].unique()[0]
    receptor = zero_crossings.query("cell == @cell")['receptor'].unique()[0]
    transmitter_mM = zero_crossings.query("cell == @cell")['transmitter_mM'].unique()[0]
    
    cells_list.append(cell)
    eq_list.append(eq)
    transmitter_list.append(transmitter)
    transmitter_mM_list.append(transmitter_mM)
    receptor_list.append(receptor)

eq_interpolated = pd.DataFrame({'cell': cells_list, 'eq_potential': eq_list, 'transmitter': transmitter_list, 'transmitter_mM': transmitter_mM_list, 'receptor': receptor_list})
# eq_interpolated.groupby(['transmitter', 'transmitter_mM', 'receptor']).eq_potential.mean()
# eq_interpolated.groupby(['transmitter', 'transmitter_mM', 'receptor']).eq_potential.sem()

In [ ]:
scale = 0.35  # scaling factor for the plot
subplot_abs_width = 1.5 * scale # width and height same
subplot_abs_spacing_width = 1.5 * scale
subplot_abs_spacing_height = 1.5 * scale
subplot_abs_excess_width = 0.3 * scale
subplot_abs_excess_height = 0.3 * scale

cols = 3
rows = 3

fig_width = (cols * subplot_abs_width) + ((cols - 1) * subplot_abs_spacing_width) + subplot_abs_excess_width
fig_height = (rows * subplot_abs_width) + ((rows - 1) * subplot_abs_spacing_height) + subplot_abs_excess_height

fig, ax = plt.subplots(3, 3, figsize=(8.3*cm_inch, 4), dpi=300)
rows = {0:{'query':"transmitter == 'Glutamate' and transmitter_mM == '50' and receptor != 'rdl_ort' and receptor != 'control'", 'zero_time': 0.17799, 'ylim':[-0.2, 1.2], 'yticks':[0, 1], 'xlim':[-0.17799, 1.04201], 'values_traces':'i_start_baseline', 'values_peak':'leak'},
        1:{'query':"transmitter == 'GABA' and transmitter_mM == '1' and receptor != 'rdl_ort'", 'zero_time': 1.2281, 'ylim':[-1,1], 'xlim':[-0.3, 1], 'yticks':[-1, 0, 1], 'values_traces':'i', 'values_peak':'i'},
        2:{'query':"transmitter == 'Histamine' and transmitter_mM == '1' and receptor != 'rdl_ort'", 'zero_time': 1.2281, 'ylim':[-3,3], 'xlim':[-0.3, 1], 'yticks':[-3, 0, 3], 'values_traces':'i', 'values_peak':'i'}}
          
for row_id in [0, 1, 2]:
    subset = averages.query(rows[row_id]['query'])
    subset.receptor = subset.receptor.astype(pd.CategoricalDtype(['rdl', 'ort', 'control'], ordered=True))
    subset = subset.sort_values('receptor')
    subset_peak_data = peak_data.query(rows[row_id]['query'])
    zero_time = rows[row_id]['zero_time']
    xlim = rows[row_id]['xlim']
    ylim = rows[row_id]['ylim']
    yticks = rows[row_id]['yticks']
    values_traces = rows[row_id]['values_traces']
    values_peak = rows[row_id]['values_peak']
    
    ax_id = 0
    for key, cell_config in subset.groupby(['transmitter', 'transmitter_mM', 'receptor'], dropna=False, observed=True):
        transmitter = key[0]
        transmitter_mM = key[1]
        receptor = key[2]
        cell_count = cell_counts.reset_index().query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")['cell_count'].values[0]

        for s in cell_config.sweep.unique():
            subsubset = cell_config.query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")
            sns.lineplot(x=subsubset.query("sweep == @s").time.values - zero_time, y=subsubset.query("sweep == @s")[values_traces] / 1000, c=colormap[receptor], linewidth=0.75, ax=ax[row_id, ax_id], alpha=0.8)
            ax[row_id, ax_id].set_title(str(key) + ' n = ' + str(cell_count))
            ax[row_id, ax_id].set_xlim(xlim)
            ax[row_id, ax_id].set_ylim(ylim)
            ax[row_id, ax_id].set_yticks(yticks)
            ax[row_id, ax_id].set_ylabel('Current (nA)', labelpad=0);
            ax[row_id, ax_id].set_xlabel('Time (s)', labelpad=2);
        ax_id = ax_id + 1
    
    ax[row_id, 2].axvline(x=0, color='black', linewidth=0.5, zorder=0)
    ax[row_id, 2].axhline(y=0, color='black', linewidth=0.5, zorder=0)
    sns.lineplot(x=subset_peak_data.v, y=subset_peak_data[values_peak] / 1000, hue=subset_peak_data.receptor, hue_order=['ort', 'rdl'], palette=colormap, ax=ax[row_id, 2], estimator='mean', errorbar='se', markers=True, legend=False, linewidth=1, err_kws={'linewidth': 0});
    ax[row_id, 2].set_xticks([-120, -80, -40, 0, 40]);
    ax[row_id, 2].set_xlim([-120, 40]);
    ax[row_id, 2].set_ylim(ylim)
    ax[row_id, 2].set_yticks(yticks)
    if values_peak == 'leak':
        ax[row_id, 2].set_ylabel('Steady-state current (nA)', labelpad=0);
    else:
        ax[row_id, 2].set_ylabel('Peak current (nA)', labelpad=0);
    ax[row_id, 2].set_xlabel('Membrane potential (mV)', labelpad=2);
    
    for i in [0,1,2]:
        ax[row_id, i].spines[['top', 'right']].set_visible(False)
        # make square
        xvals,yvals = ax[row_id, i].axes.get_xlim(),ax[row_id, i].axes.get_ylim()
        xrange = xvals[1]-xvals[0]
        yrange = yvals[1]-yvals[0]
        ax[row_id, i].set_aspect(1*(xrange/yrange), adjustable='box')
        # Show all labels
        ax[row_id, i].xaxis.set_tick_params(labelbottom=True)
        ax[row_id, i].yaxis.set_tick_params(labelleft=True)
    
    plt.subplots_adjust(wspace=0.5, hspace=0.5)

figtitle = "ephys_traces_voltage_GABA1mM_HA1mM.pdf"
plt.savefig(savepath_fig / figtitle)

In [ ]:
rows = {0:{'query':"transmitter == 'GABA' and transmitter_mM == '50' and receptor != 'rdl_ort'", 'zero_time': 1.2281, 'ylim':[-1,1], 'xlim':[-0.3, 1], 'yticks':[-1, 0, 1], 'values_traces':'i', 'values_peak':'i'},
       1:{'query':"transmitter == 'GABA' and transmitter_mM == '50' and receptor != 'rdl_ort'", 'zero_time': 1.2281, 'ylim':[-0.2, 0.7], 'xlim':[-0.01, 0.1], 'yticks':[0, 0.5], 'values_traces':'i', 'values_peak':'i'},
       }

cols = 4
rows_n = len(rows.keys())
last_col = cols - 1

fig, ax = plt.subplots(rows_n, cols, figsize=(10*cm_inch, 2.5), dpi=300, squeeze=False)

for row_id in rows.keys():
    subset = averages.query(rows[row_id]['query'])
    subset.receptor = subset.receptor.astype(pd.CategoricalDtype(['rdl', 'ort', 'control'], ordered=True))
    subset = subset.sort_values('receptor')
    subset_peak_data = peak_data.query(rows[row_id]['query'])
    zero_time = rows[row_id]['zero_time']
    xlim = rows[row_id]['xlim']
    ylim = rows[row_id]['ylim']
    yticks = rows[row_id]['yticks']
    values_traces = rows[row_id]['values_traces']
    values_peak = rows[row_id]['values_peak']
    
    ax_id = 0
    for key, cell_config in subset.groupby(['transmitter', 'transmitter_mM', 'receptor'], dropna=False, observed=True):
        transmitter = key[0]
        transmitter_mM = key[1]
        receptor = key[2]
        cell_count = cell_counts.reset_index().query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")['cell_count'].values[0]

        for s in cell_config.sweep.unique():
            subsubset = cell_config.query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")
            sns.lineplot(x=subsubset.query("sweep == @s").time.values - zero_time, y=subsubset.query("sweep == @s")[values_traces] / 1000, c=colormap[receptor], linewidth=0.75, ax=ax[row_id, ax_id], alpha=0.8)
            ax[row_id, ax_id].set_title(str(key) + ' n = ' + str(cell_count))
            ax[row_id, ax_id].set_xlim(xlim)
            ax[row_id, ax_id].set_ylim(ylim)
            ax[row_id, ax_id].set_yticks(yticks)
            ax[row_id, ax_id].set_ylabel('Current (nA)', labelpad=0);
            ax[row_id, ax_id].set_xlabel('Time (s)', labelpad=2);
        ax_id = ax_id + 1
    
    ax[row_id, last_col].axvline(x=0, color='black', linewidth=0.5)
    ax[row_id, last_col].axhline(y=0, color='black', linewidth=0.5)
    sns.lineplot(x=subset_peak_data.v, y=subset_peak_data[values_peak] / 1000, hue=subset_peak_data.receptor, palette=colormap, ax=ax[row_id, last_col], estimator='mean', errorbar='se', markers=True, legend=False, linewidth=1, err_kws={'linewidth': 0});
    ax[row_id, last_col].set_xticks([-120, -80, -40, 0, 40]);
    ax[row_id, last_col].set_xlim([-120, 40]);
    ax[row_id, last_col].set_ylim(ylim)
    ax[row_id, last_col].set_yticks(yticks)
    ax[row_id, last_col].set_ylabel('Current (nA)', labelpad=0);
    ax[row_id, last_col].set_xlabel('Membrane potential (mV)', labelpad=2);
    
    for i in [0,1,2, 3]:
        ax[row_id, i].spines[['top', 'right']].set_visible(False)
        # make square
        xvals,yvals = ax[row_id, i].axes.get_xlim(),ax[row_id, i].axes.get_ylim()
        xrange = xvals[1]-xvals[0]
        yrange = yvals[1]-yvals[0]
        ax[row_id, i].set_aspect(1*(xrange/yrange), adjustable='box')
        # Show all labels
        ax[row_id, i].xaxis.set_tick_params(labelbottom=True)
        ax[row_id, i].yaxis.set_tick_params(labelleft=True)
    
    ax[0,1].add_patch(patches.Rectangle((-0.01,-0.2), 0.11, 0.9, edgecolor='k',fill=False, zorder=100, lw=0.75))
    
    plt.subplots_adjust(wspace=0.5, hspace=0.5)

figtitle = 'ephys_traces_GABA50mM.pdf'
plt.savefig(savepath_fig / figtitle)

In [ ]:
# Decay tau calculations
data_rdl_GABA = averaged_cells_sorted.query("receptor == 'rdl' and transmitter == 'GABA' and transmitter_mM == '50' and sweep == 16 and cell not in ['096', '067', '057', '101']")
data_rdl_GABA['condition'] = '1_rdl_gaba_50mM'
data_ort_GABA = averaged_cells_sorted.query("receptor == 'ort' and transmitter == 'GABA' and transmitter_mM == '50' and sweep == 16 and cell != '104'")
data_ort_GABA['condition'] = '2_ort_gaba_50mM'
data_rdl_ort_GABA = averaged_cells_sorted.query("receptor == 'rdl_ort' and transmitter == 'GABA' and transmitter_mM == '50' and sweep == 16 and cell not in ['085', '072', '100', '084', '114']")
data_rdl_ort_GABA['condition'] = '3_rdl_ort_gaba_50mM'

calculate_tau_data = pd.concat([data_rdl_GABA, data_ort_GABA, data_rdl_ort_GABA]).reset_index(drop=True)

# Time of peak response
max_times = calculate_tau_data.loc[calculate_tau_data.query("time > 1.2 and time < 2").groupby('cell').i.idxmax()][['cell', 'time']].reset_index(drop=True)
max_times = max_times.set_index('cell')['time'].to_dict()

max_values = calculate_tau_data.loc[calculate_tau_data.query("time > 1.2 and time < 2").groupby('cell').i.idxmax()][['cell', 'i']].reset_index(drop=True)
max_values = max_values.set_index('cell')['i'].to_dict()

calculate_tau_data['max_time'] = calculate_tau_data.cell.map(max_times)
calculate_tau_data['max_i'] = calculate_tau_data.cell.map(max_values)
calculate_tau_data['i_normalized'] = calculate_tau_data['i'] / calculate_tau_data['max_i']
calculate_tau_data = calculate_tau_data.sort_values(['receptor', 'transmitter', 'cell', 'time']).reset_index(drop=True)

# Get the time point where current has dropped to 80% of peak value, after the peak (from where it usually looks exponential)
decay_80percent = {}

for cell in calculate_tau_data.cell.unique():
    cell_max_time = max_times[cell]
    cell_data = calculate_tau_data.query("cell == @cell and time > (@cell_max_time)")
    data_80percent_temp = cell_data.loc[(cell_data.i_normalized - 0.8).abs().idxmin()]
    decay_80percent[cell] = data_80percent_temp.time


# Based on https://swharden.com/blog/2020-09-24-python-exponential-fit/

def ExpDecay(x, m, t):
    return m * np.exp(-t * x)

conditions = calculate_tau_data.condition.sort_values().unique()

Tau_data = {}
rSquared_data = {}

for condition in conditions:
    cells = calculate_tau_data.query("condition == @condition").cell.unique()
    
    for cell in cells:
        cell_decay80 = decay_80percent[cell]
        cell_data = calculate_tau_data.query("cell == @cell")
        receptor = cell_data.receptor.unique()[0]
    
        cell_data_fitting = calculate_tau_data.query("cell == @cell and time > (@cell_decay80) and time < (@cell_decay80 + 1.3)")
        cell_data_fitting.time = cell_data_fitting.time - (cell_decay80)
        
        xs = cell_data_fitting.time.values[:].astype('float64')
        ys = cell_data_fitting.i_normalized.values[:].astype('float64')
                 
        # perform the fit using the function where B is 0
        p0 = (1, 10) # start with values near those we expect
        paramsB, cv = scipy.optimize.curve_fit(ExpDecay, xs, ys, p0)
        mB, tB = paramsB
    
        tauSec = (1 / tB)
        Tau_data[cell] = tauSec
        
        xs2 = xs
        ys2 = ExpDecay(xs2, mB, tB)
        xs2 = xs2 + cell_decay80
        
        # determine quality of the fit
        squaredDiffs = np.square(ys - ExpDecay(xs, mB, tB))
        squaredDiffsFromMean = np.square(ys - np.mean(ys))
        rSquared = 1 - np.sum(squaredDiffs) / np.sum(squaredDiffsFromMean)
        rSquared_data[cell] = rSquared

tau_data = calculate_tau_data[['condition', 'receptor', 'cell', 'transmitter', 'transmitter_mM']].copy().drop_duplicates().reset_index(drop=True)
tau_data = tau_data.sort_values('condition').reset_index(drop=True)
tau_data['tau'] = tau_data.cell.map(Tau_data)
tau_data['rSquared'] = tau_data.cell.map(rSquared_data)
tau_data_means = pd.DataFrame(tau_data.groupby(['condition', 'receptor', 'transmitter', 'transmitter_mM']).tau.mean())
rSquared_data_means = pd.DataFrame(tau_data.groupby(['condition', 'receptor', 'transmitter', 'transmitter_mM']).rSquared.mean())

# tau_data.groupby(['condition','receptor','transmitter','transmitter_mM']).tau.mean()
# tau_data.groupby(['condition','receptor','transmitter','transmitter_mM']).tau.std()

In [ ]:
cols = 4
rows_n = 2
last_col = cols - 1
zero_time = 1.2281

# Plot in the same subplot shape as before to combine the two later
fig, ax = plt.subplot_mosaic('ABCD;EFFG', figsize=(10*cm_inch, 2.5), dpi=300)
plt.subplots_adjust(wspace=0.5, hspace=0.5)

conditions = ['1_rdl_gaba_50mM', '2_ort_gaba_50mM']

for condition in conditions:
    cells = calculate_tau_data.query("condition == @condition").cell.unique()
    
    for cell in cells:
        cell_decay80 = decay_80percent[cell]
        cell_data = calculate_tau_data.query("cell == @cell")
        receptor = cell_data.receptor.unique()[0]
        sns.lineplot(x=cell_data.time.values - zero_time, y=cell_data['i_normalized'], ax=ax['F'], color=colormap[receptor], lw=0.5, alpha=0.5, zorder=0)
        # ax['F'].set_xlim([1.2, 3])#3.5])  
        ax['F'].set_xlim([-0.1,2.2])
        ax['F'].set_xticks([0,1,2])
        ax['F'].set_ylim([-0.1,1.1])
        ax['F'].set_yticks([0,1])
        ax['F'].set_ylabel('Norm. current (AU)', labelpad=0);
        ax['F'].set_xlabel('Time (s)', labelpad=2);

tau_data_subset = tau_data.query("condition in @conditions")
tau_data_means_subset = tau_data_means.query("condition in @conditions")
sns.stripplot(data=tau_data_subset, x='condition', y='tau', palette=colormap, hue='receptor', alpha=0.5, s=3, jitter=0.1, dodge=False, edgecolor='white', linewidth=0.2, ax=ax['G'], legend=None)
sns.scatterplot(data=tau_data_means_subset, x='condition', y='tau', palette=colormap, hue='receptor', alpha=1, ax=ax['G'], legend=None, zorder=3, marker='_', s=150, lw=1)
ax['G'].set_ylabel('Tau decay (s)');
ax['G'].set_yscale('log')
ax['G'].set_ylim([0.005, 4])
# ax['G'].set_yticks([0, 1, 2, 3])
ax['G'].set_xlabel('');
ax['G'].margins(x=0.5);

for i in 'ABCDEFG':
    ax[i].spines[['top', 'right']].set_visible(False)
    # Show all labels
    ax[i].xaxis.set_tick_params(labelbottom=True)
    ax[i].yaxis.set_tick_params(labelleft=True)

for i in 'ABCDEG':    
    # make square
    xvals,yvals = ax[i].axes.get_xlim(),ax[i].axes.get_ylim()
    if ax[i].get_yscale() == "log":
        yrange = np.log10(yvals[1]) - np.log10(yvals[0]) 
    else:
        yrange = yvals[1]-yvals[0]
    xrange = xvals[1]-xvals[0]
    ax[i].set_aspect(1*(xrange/yrange), adjustable='box')
    ax[i].set_box_aspect(1)
    
for i in 'F':
    # make rectangle
    xvals,yvals = ax[i].axes.get_xlim(),ax[i].axes.get_ylim()
    xrange = xvals[1]-xvals[0]
    yrange = yvals[1]-yvals[0]
    ax[i].set_aspect(0.4*(xrange/yrange), adjustable='box')
    
figtitle = 'tau_traces.pdf'
plt.savefig(savepath_fig / figtitle)

In [ ]:
# Number of cells in above plot
tau_data_subset.groupby('condition').receptor.value_counts()

In [ ]:
# subset of cell averages from cells included in equilibrium potential analysis (responding cells)

averaged_temp_eq = {}
for index, column_name in enumerate(['i', 'i_start_baseline', 'i_raw', 'Z_i', 'leak', 'pdes', 'i_abs', 'puff_time']):
    averaged_cells_eq = averaged_cells.query("cell in @responding_cells")
    averaged_temp_eq[column_name] = averaged_cells_eq.groupby(['time', 'v', 'transmitter', 'transmitter_mM', 'receptor', 'sweep', 'protocol'])[column_name].mean()
    averaged_temp_eq['n_cells'] = averaged_cells_eq.groupby(['time', 'v', 'transmitter', 'transmitter_mM', 'receptor', 'sweep', 'protocol']).cell.count()
averages_eq = pd.DataFrame(averaged_temp_eq).reset_index()

In [ ]:
eq_histamine = eq_interpolated.query("transmitter_mM == '1' and receptor == 'ort' and not (receptor == 'ort' and transmitter == 'GABA')").reset_index(drop=True)
eq_histamine_means = pd.DataFrame(eq_histamine.groupby(['transmitter']).eq_potential.mean()).reset_index()
eq_GABA = eq_interpolated.query("transmitter_mM == '1' and receptor == 'rdl'").reset_index(drop=True)
eq_GABA_means = pd.DataFrame(eq_GABA.groupby(['transmitter']).eq_potential.mean()).reset_index()

In [ ]:
fig, ax = plt.subplots(2, 4, squeeze=False, figsize=(14*cm_inch, 3), width_ratios=[1,1,1,0.5], dpi=300)
plt.subplots_adjust(wspace=0.5, hspace=0.5)

query = "transmitter in ['GABA', 'GABA_reducedCl'] and transmitter_mM == '1' and receptor == 'rdl'"
subset = averages_eq.query(query)
subset_peak_data = peak_data.query(query).query("cell in @responding_cells").drop(columns=['protocol'])

ax_x = 0
for key, cell_config in subset.groupby(['transmitter', 'transmitter_mM', 'receptor'], dropna=False):
    transmitter = key[0]
    transmitter_mM = key[1]
    receptor = key[2]
    n_cells = cell_config.n_cells.unique()
    
    for s in cell_config.sweep.unique():
        subsubset = cell_config.query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")
        sns.lineplot(x=subsubset.query("sweep == @s").time.values - 1.2281, y=subsubset.query("sweep == @s").i / 1000, c=colormap_rdl_reducedCl[transmitter], linewidth=0.75, ax=ax[0, ax_x], alpha=0.8)
        ax[0,ax_x].set_title(str(key) + ' n = ' + str(n_cells))
        ax[0,ax_x].set_xlim([-0.3, 1])
        ax[0,ax_x].set_ylim([-1.1, 1.1])
        ax[0,ax_x].set_yticks([-1, 0, 1])
        ax[0,ax_x].set_ylabel('Current (nA)');
        ax[0,ax_x].set_xlabel('Time (s)');  
    ax_x = ax_x + 1 

ax[0,2].axvline(x=0, color='black', linewidth=0.5)
ax[0,2].axhline(y=0, color='black', linewidth=0.5)
sns.lineplot(data=subset_peak_data, x='v', y='i', hue='transmitter', palette=colormap_rdl_reducedCl, ax=ax[0,2], estimator='mean', errorbar='se', markers=True, legend=False, linewidth=1, err_kws={'linewidth': 0});
ax[0,2].annotate('x', (reversal_Cl_normal,0), c=colormap_rdl_reducedCl['GABA'])
ax[0,2].annotate('{} mV'.format(np.format_float_positional(reversal_Cl_normal, precision=1)), (reversal_Cl_normal,300), c=colormap_rdl_reducedCl['GABA'])
ax[0,2].annotate('x', (reversal_Cl_reduced,0), c=colormap_rdl_reducedCl['GABA_reducedCl'])
ax[0,2].annotate('{} mV'.format(np.format_float_positional(reversal_Cl_reduced, precision=1)), (reversal_Cl_reduced,300), c=colormap_rdl_reducedCl['GABA_reducedCl'])
ax[0,2].set_xticks([-120,-80, -40, 0, 40]);
ax[0,2].set_ylim([-1100, 1100]);
ax[0,2].set_yticks([-1000, 0, 1000]);
ax[0,2].set_yticklabels([-1, 0, 1]);
ax[0,2].set_xlim([-121, 40]);
ax[0,2].set_ylabel('Current (nA)');
ax[0,2].set_xlabel('Membrane potential (mV)');

sns.stripplot(data=eq_GABA, y='eq_potential', x='transmitter', order=['GABA', 'GABA_reducedCl'], palette=colormap_rdl_reducedCl, hue='transmitter', alpha=0.5, s=3, jitter=0.3, dodge=False, edgecolor='white', linewidth=0.5, ax=ax[0,3])
sns.scatterplot(data=eq_GABA_means, y='eq_potential', x='transmitter', palette=colormap_rdl_reducedCl, hue='transmitter', alpha=1, ax=ax[0,3], legend=None, zorder=3, marker='_', s=150, lw=1.5)
ax[0,3].set_ylabel('Reversal potential (mV)');
ax[0,3].set_yticks([-120, -80, -40, 0, 40]);
ax[0,3].set_xlabel('');
ax[0,3].margins(x=0.2);

query = "transmitter in ['Histamine', 'Histamine_reducedCl'] and transmitter_mM == '1' and receptor == 'ort'"
subset = averages_eq.query(query)
subset_peak_data = peak_data.query(query).query("cell in @responding_cells").drop(columns=['protocol'])

ax_x = 0
for key, cell_config in subset.groupby(['transmitter', 'transmitter_mM', 'receptor'], dropna=False):
    transmitter = key[0]
    transmitter_mM = key[1]
    receptor = key[2]
    n_cells = cell_config.n_cells.unique()
    
    for s in cell_config.sweep.unique():
        subsubset = cell_config.query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")
        sns.lineplot(x=subsubset.query("sweep == @s").time.values - 1.2281, y=subsubset.query("sweep == @s").i / 1000, c=colormap_ort_reducedCl[transmitter], linewidth=0.75, ax=ax[1, ax_x], alpha=0.8)
        ax[1,ax_x].set_title(str(key) + ' n = ' + str(n_cells))
        ax[1,ax_x].set_xlim([-0.3, 1])
        ax[1,ax_x].set_ylim([-3, 3])
        ax[1,ax_x].set_yticks([-3, -2, -1, 0, 1, 2, 3])
        ax[1,ax_x].set_ylabel('Current (nA)');
        ax[1,ax_x].set_xlabel('Time (s)');
    # ax[ax_id].axvline(x=0, color='grey', linewidth=0.5)    
    ax_x = ax_x + 1 

ax[1,2].axvline(x=0, color='black', linewidth=0.5)
ax[1,2].axhline(y=0, color='black', linewidth=0.5)
sns.lineplot(data=subset_peak_data, x='v', y='i', hue='transmitter', palette=colormap_ort_reducedCl, ax=ax[1,2], estimator='mean', errorbar='se', markers=True, legend=False, linewidth=1, err_kws={'linewidth': 0});
ax[1,2].annotate('x', (reversal_Cl_normal,0), c=colormap_ort_reducedCl['Histamine'])
ax[1,2].annotate('{} mV'.format(np.format_float_positional(reversal_Cl_normal, precision=1)), (reversal_Cl_normal,300), c=colormap_ort_reducedCl['Histamine'])
ax[1,2].annotate('x', (reversal_Cl_reduced,0), c=colormap_ort_reducedCl['Histamine_reducedCl'])
ax[1,2].annotate('{} mV'.format(np.format_float_positional(reversal_Cl_reduced, precision=1)), (reversal_Cl_reduced,300), c=colormap_ort_reducedCl['Histamine_reducedCl'])
ax[1,2].set_xticks([-120,-80, -40, 0, 40]);
ax[1,2].set_ylim([-3000, 3000]);
ax[1,2].set_yticks([-3000, -2000, -1000, 0, 1000, 2000, 3000]);
ax[1,2].set_yticklabels([-3, -2, -1, 0, 1, 2, 3]);
ax[1,2].set_xlim([-121, 40]);
ax[1,2].set_ylabel('Current (nA)');
ax[1,2].set_xlabel('Membrane potential (mV)');

sns.stripplot(data=eq_histamine, y='eq_potential', x='transmitter', order=['Histamine', 'Histamine_reducedCl'], palette=colormap_ort_reducedCl, hue='transmitter', alpha=0.5, s=3, jitter=0.3, dodge=False, edgecolor='white', linewidth=0.5, ax=ax[1,3])
sns.scatterplot(data=eq_histamine_means, y='eq_potential', x='transmitter', palette=colormap_ort_reducedCl, hue='transmitter', alpha=1, ax=ax[1,3], legend=None, zorder=3, marker='_', s=150, lw=1.5)
ax[1,3].set_ylabel('Reversal potential (mV)');
ax[1,3].set_yticks([-120, -80, -40, 0, 40]);
ax[1,3].set_xlabel('');
ax[1,3].margins(x=0.2);

for i in [0,1,2,3]:
    ax[0,i].spines[['top', 'right']].set_visible(False)
    ax[1,i].spines[['top', 'right']].set_visible(False)

figtitle = 'ephys_reducedCl.pdf'
plt.savefig(savepath_fig / figtitle)

In [ ]:
scale = 0.35  # scaling factor for the plot
subplot_abs_width = 1.5 * scale # width and height same
subplot_abs_spacing_width = 1.5 * scale
subplot_abs_spacing_height = 1.5 * scale
subplot_abs_excess_width = 0.3 * scale
subplot_abs_excess_height = 0.3 * scale

cols = 4
rows = 4

fig_width = (cols * subplot_abs_width) + ((cols - 1) * subplot_abs_spacing_width) + subplot_abs_excess_width
fig_height = (rows * subplot_abs_width) + ((rows - 1) * subplot_abs_spacing_height) + subplot_abs_excess_height

fig, ax = plt.subplots(3, 4, figsize=(14*cm_inch, 4), dpi=300)
rows = {0:{'query':"transmitter == 'Glutamate' and transmitter_mM == '50' and receptor != 'control'", 'zero_time': 1.2281, 'ylim':[-1,1], 'yticks':[-1, 0, 1], 'xlim':[-0.3, 1], 'values_traces':'i', 'values_peak':'i'}}

for row_id in [0]:
    subset = averages.query(rows[row_id]['query'])
    subset.receptor = subset.receptor.astype(pd.CategoricalDtype(['rdl', 'ort', 'rdl_ort', 'control'], ordered=True))
    subset = subset.sort_values('receptor')
    subset_peak_data = peak_data.query(rows[row_id]['query'])
    zero_time = rows[row_id]['zero_time']
    xlim = rows[row_id]['xlim']
    ylim = rows[row_id]['ylim']
    yticks = rows[row_id]['yticks']
    values_traces = rows[row_id]['values_traces']
    values_peak = rows[row_id]['values_peak']
    
    ax_id = 0
    for key, cell_config in subset.groupby(['transmitter', 'transmitter_mM', 'receptor'], dropna=False, observed=True):
        transmitter = key[0]
        transmitter_mM = key[1]
        receptor = key[2]
        cell_count = cell_counts.reset_index().query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")['cell_count'].values[0]

        for s in cell_config.sweep.unique():
            subsubset = cell_config.query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")
            sns.lineplot(x=subsubset.query("sweep == @s").time.values - zero_time, y=subsubset.query("sweep == @s")[values_traces] / 1000, c=colormap[receptor], linewidth=0.75, ax=ax[row_id, ax_id], alpha=0.8)
            ax[row_id, ax_id].set_title(str(key) + ' n = ' + str(cell_count))
            ax[row_id, ax_id].set_xlim(xlim)
            ax[row_id, ax_id].set_ylim(ylim)
            ax[row_id, ax_id].set_yticks(yticks)
            ax[row_id, ax_id].set_ylabel('Current (nA)', labelpad=0);
            ax[row_id, ax_id].set_xlabel('Time (s)', labelpad=2);
        ax_id = ax_id + 1
    
    ax[row_id, 3].axvline(x=0, color='black', linewidth=0.5, zorder=0)
    ax[row_id, 3].axhline(y=0, color='black', linewidth=0.5, zorder=0)
    sns.lineplot(x=subset_peak_data.v, y=subset_peak_data[values_peak] / 1000, hue=subset_peak_data.receptor, hue_order=['ort', 'rdl'], palette=colormap, ax=ax[row_id, 3], estimator='mean', errorbar='se', markers=True, legend=False, linewidth=1, err_kws={'linewidth': 0});
    ax[row_id, 3].set_xticks([-120, -80, -40, 0, 40]);
    ax[row_id, 3].set_xlim([-120, 40]);
    ax[row_id, 3].set_ylim(ylim)
    ax[row_id, 3].set_yticks(yticks)
    ax[row_id, 3].set_ylabel('Current (nA)', labelpad=0);
    ax[row_id, 3].set_xlabel('Membrane potential (mV)', labelpad=2);

extra_panels = {1:{'query':"transmitter == 'GABA' and transmitter_mM == '1' and receptor == 'rdl_ort'", 'zero_time': 1.2281, 'ylim':[-1.6,1.6], 'yticks':[-1, 0, 1], 'xlim':[-0.3, 1], 'values_traces':'i', 'values_peak':'i'},
                2:{'query':"transmitter == 'Glutamate' and transmitter_mM == '50' and receptor == 'rdl_ort'", 'zero_time': 0.17799, 'ylim':[-0.2, 0.8], 'yticks':[0, 0.5], 'xlim':[-0.17799, 1.04201], 'values_traces':'i_start_baseline', 'values_peak':'leak'}, 
                3:{'query':"transmitter == 'GABA' and transmitter_mM == '50' and receptor == 'rdl_ort'", 'zero_time': 1.2281, 'ylim':[-1.6,1], 'yticks':[-1, 0, 1], 'xlim':[-0.3, 1], 'values_traces':'i', 'values_peak':'i'},
                4:{'query':"transmitter == 'Histamine' and transmitter_mM == '1' and receptor == 'rdl_ort'", 'zero_time': 1.2281, 'ylim':[-1,1], 'yticks':[-1, 0, 1], 'xlim':[-0.3, 1], 'values_traces':'i', 'values_peak':'i'},
               }

for extra_panel_id in [1, 2, 3, 4]:
    subset = averages.query(extra_panels[extra_panel_id]['query'])
    subset.receptor = subset.receptor.astype(pd.CategoricalDtype(['rdl', 'ort', 'rdl_ort', 'control'], ordered=True))
    subset = subset.sort_values('receptor')
    subset_peak_data = peak_data.query(extra_panels[extra_panel_id]['query'])
    zero_time = extra_panels[extra_panel_id]['zero_time']
    xlim = extra_panels[extra_panel_id]['xlim']
    ylim = extra_panels[extra_panel_id]['ylim']
    yticks = extra_panels[extra_panel_id]['yticks']
    values_traces = extra_panels[extra_panel_id]['values_traces']
    values_peak = extra_panels[extra_panel_id]['values_peak']

    ax_id = 1
    for key, cell_config in subset.groupby(['transmitter', 'transmitter_mM', 'receptor'], dropna=False, observed=True):
        transmitter = key[0]
        transmitter_mM = key[1]
        receptor = key[2]
        cell_count = cell_counts.reset_index().query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")['cell_count'].values[0]
        if extra_panel_id == 1:
            row_id = 1
            ax_id = 0
        elif extra_panel_id == 2:
            row_id = 1
            ax_id = 2
        elif extra_panel_id == 3:
            row_id = 2
            ax_id = 0
        else:
            row_id = 2
            ax_id = 2
        
        for s in cell_config.sweep.unique():
            subsubset = cell_config.query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")
            sns.lineplot(x=subsubset.query("sweep == @s").time.values - zero_time, y=subsubset.query("sweep == @s")[values_traces] / 1000, c=colormap[receptor], linewidth=0.75, ax=ax[row_id, ax_id], alpha=0.8)
            ax[row_id, ax_id].set_title(str(key) + ' n = ' + str(cell_count))
            ax[row_id, ax_id].set_xlim(xlim)
            ax[row_id, ax_id].set_ylim(ylim)
            ax[row_id, ax_id].set_yticks(yticks)
            ax[row_id, ax_id].set_ylabel('Current (nA)', labelpad=0);
            ax[row_id, ax_id].set_xlabel('Time (s)', labelpad=2);

        ax[row_id, ax_id+1].axvline(x=0, color='black', linewidth=0.5, zorder=0)
        ax[row_id, ax_id+1].axhline(y=0, color='black', linewidth=0.5, zorder=0)
        sns.lineplot(x=subset_peak_data.v, y=subset_peak_data[values_peak] / 1000, hue=subset_peak_data.receptor, hue_order=['ort', 'rdl', 'rdl_ort', 'control'], palette=colormap, ax=ax[row_id, ax_id+1], estimator='mean', errorbar='se', markers=True, legend=False, linewidth=1, err_kws={'linewidth': 0});
        ax[row_id, ax_id+1].set_xticks([-120, -80, -40, 0, 40]);
        ax[row_id, ax_id+1].set_xlim([-120, 40]);
        ax[row_id, ax_id+1].set_ylim(ylim)
        ax[row_id, ax_id+1].set_yticks(yticks)
        if values_peak == 'leak':
            ax[row_id, ax_id+1].set_ylabel('Current (nA)', labelpad=0);
        else:
            ax[row_id, ax_id+1].set_ylabel('Current (nA)', labelpad=0);
        ax[row_id, ax_id+1].set_xlabel('Membrane potential (mV)', labelpad=2);
 
for i in np.arange(0,4):
    for j in np.arange(0,3):
        ax[j, i].spines[['top', 'right']].set_visible(False)
        # make square
        xvals,yvals = ax[j, i].axes.get_xlim(),ax[j, i].axes.get_ylim()
        xrange = xvals[1]-xvals[0]
        yrange = yvals[1]-yvals[0]
        ax[j, i].set_aspect(1*(xrange/yrange), adjustable='box')
        # Show all labels
        ax[j, i].xaxis.set_tick_params(labelbottom=True)
        ax[j, i].yaxis.set_tick_params(labelleft=True)

plt.subplots_adjust(wspace=0.5, hspace=0.5)

figtitle = 'ephys_traces_glutamate_rdlort.pdf'
plt.savefig(savepath_fig / figtitle)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(3, 2), dpi=300, squeeze=False)

query ="transmitter == 'GABA' and transmitter_mM == '50' and receptor == 'control'"
zero_time = 0.17799, 
ylim = [-0.2, 0.8]
yticks = [0, 0.5]
xlim =[-0.17799, 1.04201]
values_traces = 'i_start_baseline'
values_peak = 'leak'

subset = averages.query(query)
subset.receptor = subset.receptor.astype(pd.CategoricalDtype(['rdl', 'ort', 'rdl_ort', 'control'], ordered=True))
subset = subset.sort_values('receptor')
subset_peak_data = peak_data.query(query)

for key, cell_config in subset.groupby(['transmitter', 'transmitter_mM', 'receptor'], dropna=False, observed=True):
    transmitter = key[0]
    transmitter_mM = key[1]
    receptor = key[2]
    cell_count = cell_counts.reset_index().query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")['cell_count'].values[0]

    row_id = 0
    ax_id = 0
    for s in cell_config.sweep.unique():
        subsubset = cell_config.query("transmitter == @transmitter and transmitter_mM == @transmitter_mM and receptor == @receptor")
        sns.lineplot(x=subsubset.query("sweep == @s").time.values - zero_time, y=subsubset.query("sweep == @s")[values_traces] / 1000, c=colormap[receptor], linewidth=0.75, ax=ax[row_id, ax_id], alpha=0.8)
        ax[row_id, ax_id].set_title(str(key) + ' n = ' + str(cell_count))
        ax[row_id, ax_id].set_xlim(xlim)
        ax[row_id, ax_id].set_ylim(ylim)
        ax[row_id, ax_id].set_yticks(yticks)
        ax[row_id, ax_id].set_ylabel('Current (nA)', labelpad=0);
        ax[row_id, ax_id].set_xlabel('Time (s)', labelpad=2);

    ax[row_id, ax_id+1].axvline(x=0, color='black', linewidth=0.5, zorder=0)
    ax[row_id, ax_id+1].axhline(y=0, color='black', linewidth=0.5, zorder=0)
    sns.lineplot(x=subset_peak_data.v, y=subset_peak_data[values_peak] / 1000, hue=subset_peak_data.receptor, hue_order=['ort', 'rdl', 'rdl_ort', 'control'], palette=colormap, ax=ax[row_id, ax_id+1], estimator='mean', errorbar='se', markers=True, legend=False, linewidth=1, err_kws={'linewidth': 0});
    ax[row_id, ax_id+1].set_xticks([-120, -80, -40, 0, 40]);
    ax[row_id, ax_id+1].set_xlim([-120, 40]);
    ax[row_id, ax_id+1].set_ylim(ylim)
    ax[row_id, ax_id+1].set_yticks(yticks)
    ax[row_id, ax_id+1].set_ylabel('Current (nA)', labelpad=0);
    ax[row_id, ax_id+1].set_xlabel('Membrane potential (mV)', labelpad=2);
 
for i in [0,1]:
    ax[0,i].spines[['top', 'right']].set_visible(False)
    # make square
    xvals,yvals = ax[0,i].axes.get_xlim(),ax[0,i].axes.get_ylim()
    xrange = xvals[1]-xvals[0]
    yrange = yvals[1]-yvals[0]
    ax[0,i].set_aspect(1*(xrange/yrange), adjustable='box')
    # Show all labels
    ax[0,i].xaxis.set_tick_params(labelbottom=True)
    ax[0,i].yaxis.set_tick_params(labelleft=True)

plt.subplots_adjust(wspace=0.5, hspace=0.5)

figtitle = 'ephys_traces_controlvolt.pdf'
plt.savefig(savepath_fig / figtitle)

In [ ]:
####### CSV OUTPUTS ########

# For modeling
voltage_response_values = peak_data.query("transmitter == 'Glutamate' and transmitter_mM == '50' and receptor == 'rdl'")
voltage_response_values = pd.DataFrame(voltage_response_values.groupby('v').leak.mean()) / 1000
voltage_response_values = voltage_response_values.reset_index().rename(columns={'v': 'holding_potential_mV','leak':'steady_state_current_nA'})
voltage_response_values.to_csv(savepath_data / 'ephys_rdl_IV_steady_state_current_nA.csv', sep=',', float_format='%.5f')

# Rdl voltage response trace for model
temp_voltage = averages.query("transmitter == 'Glutamate' and transmitter_mM == '50' and receptor == 'rdl' and v == 40 and time < 1")
temp_voltage['time_from_vstep'] = temp_voltage.time - 0.17799

# Remove capacitive current
rm_cap_curr = temp_voltage.query("time < 0.17790 or time > 0.17950")
rm_cap_curr['i_start_baseline_normalized'] = rm_cap_curr['i_start_baseline'] / rm_cap_curr['i_start_baseline'].max()

current_trace = temp_voltage.reset_index(drop=True)[['time_from_vstep', 'i_start_baseline']].rename(columns={'i_start_baseline': 'current_nA'})
current_trace['current_nA'] = current_trace['current_nA'] / 1000
current_trace.to_csv(savepath_data / 'rdl_40mV_current_response_nA.csv', sep=',', float_format='%.5f')
current_trace_no_capacitive = rm_cap_curr.reset_index(drop=True)[['time_from_vstep', 'i_start_baseline', 'i_start_baseline_normalized']].rename(columns={'i_start_baseline': 'current_nA', 'i_start_baseline_normalized': 'current_normalized'})
current_trace_no_capacitive['current_nA'] = current_trace_no_capacitive['current_nA'] / 1000
current_trace_no_capacitive.to_csv(savepath_data / 'rdl_40mV_current_response_nA_no_capacitive.csv', sep=',', float_format='%.5f')

response_ratios = peak_data.query("transmitter == 'GABA' and v == 40")
response_ratios['ratio_voltage_ligand_response'] = response_ratios.leak / response_ratios.i
response_ratios['percent_voltage_full_response'] = response_ratios.leak / (response_ratios.leak + response_ratios.i)
# response_ratios.query("transmitter_mM == '1' and receptor == 'rdl' and cell in @responding_cells").ratio_voltage_ligand_response.mean()
# response_ratios.query("transmitter_mM == '1' and receptor == 'rdl' and cell in @responding_cells").percent_voltage_full_response.mean()

In [ ]:
####### For statistical tests ########
# Equilibrium potential
eq_GABA_wide = eq_GABA.pivot(index='cell', columns='transmitter', values='eq_potential').reset_index().drop(columns='cell')
eq_GABA_wide = eq_GABA_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
eq_GABA_wide.columns.name = None
eq_GABA_wide = eq_GABA_wide.rename(columns={'GABA':'eq_mV_normal_Cl-', 'GABA_reducedCl':'eq_mV_low_Cl-'})
eq_GABA_wide.to_csv(savepath_data / 'ephys_eq_rdl_reducedCl_GABA1mM_mV.csv', sep=',', float_format='%.5f')

eq_histamine_wide = eq_histamine.pivot(index='cell', columns='transmitter', values='eq_potential').reset_index().drop(columns='cell')
eq_histamine_wide = eq_histamine_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
eq_histamine_wide.columns.name = None
eq_histamine_wide = eq_histamine_wide.rename(columns={'Histamine':'eq_mV_normal_Cl-', 'Histamine_reducedCl':'eq_mV_low_Cl-'})
eq_histamine_wide.to_csv(savepath_data / 'ephys_eq_ort_reducedCl_histamine1mM_mV.csv', sep=',', float_format='%.5f')

tau_data_wide = tau_data.query("condition in ['1_rdl_gaba_50mM', '2_ort_gaba_50mM']").pivot(index='cell', columns='condition', values='tau').reset_index().drop(columns='cell')
tau_data_wide = tau_data_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
tau_data_wide.columns.name = None
tau_data_wide = tau_data_wide.rename(columns={'1_rdl_gaba_50mM':'tau_s_rdl_GABA50mM', '2_ort_gaba_50mM':'tau_s_ort_GABA50mM'})
tau_data_wide.to_csv(savepath_data / 'ephys_tau_decay_rdl_ort_GABA50mM_mV.csv', sep=',', float_format='%.5f')